# **Versión 2: Dataset balanceado**

1. Random Forest
2. XGBoost

In [ ]:
import re
import pandas as pd
import joblib
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

drive.mount("/content/gdrive")

%cd "/content/gdrive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales"

df_raw = pd.read_csv('df_combined.csv', delimiter=',', encoding='latin1')
df_raw.head()

df_cpp = df_raw.copy()

Mounted at /content/gdrive
/content/gdrive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales


## 1. Modificar la función de preprocesamiento

In [ ]:
def preprocess_code(code):
    # 1. Definir palabras clave de C++ que NO queremos enmascarar
    protected_keywords = {
        'int', 'float', 'double', 'char', 'bool', 'void', 'if', 'else',
        'for', 'while', 'do', 'return', 'switch', 'case', 'break',
        'continue', 'include', 'main', 'cout', 'cin', 'endl', 'printf',
        'scanf', 'vector', 'string', 'std', 'namespace', 'class', 'struct'
    }

    # Limpieza básica de comentarios y cadenas (como ya lo haces)
    code = re.sub(r'//.*', '', code)
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'".*?"', 'STR_LITERAL', code)
    code = re.sub(r'\d+', 'NUM_LITERAL', code)

    # 2. Modificar el enmascaramiento de identificadores
    # Usamos una función en re.sub para verificar si la palabra está en nuestra lista
    def replace_identifier(match):
        word = match.group(0)
        return word if word in protected_keywords else 'IDENTIFIER'

    # Buscamos palabras que empiecen con letra o guion bajo
    code = re.sub(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', replace_identifier, code)

    return code

## 2. Preprocesamiento de datos

In [ ]:
df_cpp['cleaned_code'] = df_cpp['code'].astype(str).apply(preprocess_code)

In [ ]:
X = df_cpp["cleaned_code"]
y = df_cpp["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## **Versión 2.1: Random Forest**

In [ ]:
model_rf = Pipeline([
    ("tfidf", TfidfVectorizer(
        # Analizar caracteres en lugar de palabras
        analyzer='char',
        # Rango de 3 a 6 caracteres para capturar estructuras sintácticas
        ngram_range=(3, 6),
        # Aumentamos las características porque los n-gramas de char generan muchos términos
        max_features=10000,
        lowercase=False #En C++, 'Vector' e 'vector' pueden ser cosas distintas
    )),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model_rf.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char', lowercase=False,
                                 max_features=10000, ngram_range=(3, 6))),
                ('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)

    print(f"Accuracy {model_name}:", accuracy_score(y_test, y_pred))
    print(f"\nMatriz de confusión {model_name}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nReporte {model_name}:")
    print(classification_report(y_test, y_pred))

evaluate_model(model_rf, X_test, y_test, model_name="RF")

Accuracy RF: 0.9559932942162616

Matriz de confusión RF:
[[1106   82]
 [  23 1175]]

Reporte RF:
              precision    recall  f1-score   support

           0       0.98      0.93      0.95      1188
           1       0.93      0.98      0.96      1198

    accuracy                           0.96      2386
   macro avg       0.96      0.96      0.96      2386
weighted avg       0.96      0.96      0.96      2386



## **Versión 2.2: XGBoost**

In [ ]:
from xgboost import XGBClassifier

model_xgb = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 6),
        max_features=10000,
        lowercase=False
    )),
    ("xgb", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6
    ))
])

model_xgb.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char', lowercase=False,
                                 max_features=10000, ngram_range=(3, 6))),
                ('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric=None,
                               feature_types=None, featur...s=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [ ]:
evaluate_model(model_xgb, X_test, y_test, model_name="XGB")

Accuracy XGB: 0.9694048616932104

Matriz de confusión XGB:
[[1126   62]
 [  11 1187]]

Reporte XGB:
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      1188
           1       0.95      0.99      0.97      1198

    accuracy                           0.97      2386
   macro avg       0.97      0.97      0.97      2386
weighted avg       0.97      0.97      0.97      2386

